# Database Construction: VP-DB

Build the VibroPredict Database (VP-DB) by:
1. Loading KinHub-27k as the gold-standard core
2. Augmenting with EnzyExtractDB entries
3. Standardizing kinetic parameters and chemical representations
4. Splitting into train/val/test sets

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path

from vibropredict.data.kinhub import KinHubLoader
from vibropredict.data.enzyextract import EnzyExtractFilter
from vibropredict.data.standardization import (
    log_transform_kcat, canonicalize_smiles, cluster_split
)

## Step 1: Load KinHub-27k

In [ ]:
# Download KinHub-27k CSV first (not included in repo)
KINHUB_PATH = '../data/vibropredict/kinhub_27k.csv'

loader = KinHubLoader(KINHUB_PATH)
kinhub_df = loader.load()
kinhub_df = loader.validate(kinhub_df)
kinhub_df = loader.resolve_ambiguities(kinhub_df)
print(f'KinHub entries after cleaning: {len(kinhub_df)}')

## Step 2: Augment with EnzyExtractDB

In [ ]:
ENZYEXTRACT_PATH = '../data/vibropredict/enzyextract.csv'

extractor = EnzyExtractFilter(ENZYEXTRACT_PATH)
enzext_df = extractor.load()
enzext_filtered = extractor.filter(enzext_df, set(kinhub_df['uniprot_id']))
combined_df = extractor.merge(kinhub_df, enzext_filtered)
print(f'Combined entries: {len(combined_df)}')

## Step 3: Standardize

In [ ]:
combined_df = log_transform_kcat(combined_df)
combined_df = canonicalize_smiles(combined_df)

print(f'log_kcat range: [{combined_df["log_kcat"].min():.2f}, {combined_df["log_kcat"].max():.2f}]')
print(f'Missing SMILES: {combined_df["substrate_smiles"].isna().sum()}')

## Step 4: Split

In [ ]:
train_df, val_df, test_df = cluster_split(combined_df)
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

# Save
output_dir = Path('../data/vibropredict/splits')
output_dir.mkdir(parents=True, exist_ok=True)
train_df.to_csv(output_dir / 'train.csv', index=False)
val_df.to_csv(output_dir / 'val.csv', index=False)
test_df.to_csv(output_dir / 'test.csv', index=False)
print('Saved splits to', output_dir)